In [1]:
#IMPORTING stuff including my online repo with useful tools for processing chess files (PGN)

import sys
import os, sys
!pip install chess mlflow
if os.path.exists("/kaggle/working/ludus_scacchi"):
    print("Repository already exists, skipping clone.")
    sys.path.append("/kaggle/working/ludus_scacchi")
else:
    !git clone https://github.com/paolomostardi/ludus_scacchi.git /kaggle/working/ludus_scacchi
    sys.path.append("/kaggle/working/ludus_scacchi")
from pathlib import Path
from Backend.data_pipeline import from_PGN_generate_bitboards
from Backend.data_pipeline import gm_pipeline as pipe
import pandas as pd
import numpy as np
import keras
import mlflow
import mlflow.keras

MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "file:///kaggle/working/mlruns")
if MLFLOW_TRACKING_URI.startswith("file://"):
    Path(MLFLOW_TRACKING_URI.replace("file://", "")).mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("fit-model-notebook")
mlflow.keras.autolog(log_models=True)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 43.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.5/829.5 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 16.2 MB/s eta 0:00:00
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256

2026-03-14 19:30:03.665483: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773516603.817689      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773516603.859300      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773516604.198242      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773516604.198285      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773516604.198287      17 computation_placer.cc:177] computation placer alr

In [2]:
from Backend.model_architecture.implementation.experimental.model_testing_mlflow import model_testing_mlflow

model = model_testing_mlflow(conv_filters=12, upsample_factor=4)

2026-03-14 19:30:21.918007: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [3]:
!rm -r ludus_scacchi

In [4]:
from keras.models import load_model

### Current accuracy (in training: 0.52) - trained on chunk 3 - I did skip chunk 1 i belive because of some mix up, with indexes.



In [5]:
# how many cycles need to be done
# used to avoid timeout with heavier models 

MAX_N = 5
n = 5
starting_n = 0
conv_size = 32

model_name = 'lenet_baseline'
year_month = '2013-01'

mlflow_run_id = None

with mlflow.start_run(run_name=f"{model_name}_{year_month}") as run:
    mlflow_run_id = run.info.run_id
    mlflow.log_params({
        "model_name": model_name,
        "year_month": year_month,
        "max_n": MAX_N,
        "start_chunk": starting_n,
        "end_chunk": n - 1,
        "conv_size": conv_size,
        "kernel_size": (2, 2),
        "epochs_per_chunk": 5,
        "batch_size": 64
    })

    for i in range(starting_n, n):
        print("=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=")
        print("Starting training with chunk number: ", i)
        
        model.name = model_name + '_chunk_' + str(i) + '_' + year_month
        x = np.load('/kaggle/input/lichess-bitboards-2013-01/chunk_' + str(i) + '.npy')
        y = np.load('/kaggle/input/lichess-bitboards-2013-01/chunk_' + str(i) + '_y.npy')

        # leaving space for evaluation 
        if i == MAX_N - 1:
            x = x[:-10000]
            y = y[:-10000]

        x = np.transpose(x, (0, 2, 3, 1))
        
        y2 = []
        for j in y:
            y2.append(j[0].flat)
        y2 = np.array(y2)
        
        history = model.fit(x, y2, batch_size=64, epochs=5)
        mlflow.log_metric("chunk_index", i, step=i)
        model.save(model.name + '.keras')
        print("saved model with name: ", model.name)
    model.save(f'{model_name}.keras')


=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=
Starting training with chunk number:  0


Epoch 1/5
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 820s 52ms/step - accuracy: 0.0738 - loss: 3.5675
Epoch 2/5
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 819s 52ms/step - accuracy: 0.2482 - loss: 2.7386
Epoch 3/5
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 831s 53ms/step - accuracy: 0.3342 - loss: 2.4456
Epoch 4/5
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 830s 53ms/step - accuracy: 0.3672 - loss: 2.3009
Epoch 5/5
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 831s 53ms/step - accuracy: 0.3825 - loss: 2.2163


2026/03/14 20:39:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


saved model with name:  lenet_baseline_chunk_0_2013-01
=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=
Starting training with chunk number:  1


Epoch 1/5
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 802s 51ms/step - accuracy: 0.3920 - loss: 2.1705
Epoch 2/5
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 836s 53ms/step - accuracy: 0.3995 - loss: 2.1259
Epoch 3/5
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 839s 54ms/step - accuracy: 0.4056 - loss: 2.0954
Epoch 4/5
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 847s 54ms/step - accuracy: 0.4080 - loss: 2.0762
Epoch 5/5
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 844s 54ms/step - accuracy: 0.4110 - loss: 2.0561


...

2026/03/15 00:13:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


saved model with name:  lenet_baseline_chunk_3_2013-01
=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=
Starting training with chunk number:  4


Epoch 1/5
15469/15469 ━━━━━━━━━━━━━━━━━━━━ 865s 56ms/step - accuracy: 0.4229 - loss: 1.9608
Epoch 2/5
15469/15469 ━━━━━━━━━━━━━━━━━━━━ 930s 60ms/step - accuracy: 0.4247 - loss: 1.9476
Epoch 3/5
15469/15469 ━━━━━━━━━━━━━━━━━━━━ 937s 61ms/step - accuracy: 0.4266 - loss: 1.9378
Epoch 4/5
15469/15469 ━━━━━━━━━━━━━━━━━━━━ 943s 61ms/step - accuracy: 0.4265 - loss: 1.9307
Epoch 5/5
15469/15469 ━━━━━━━━━━━━━━━━━━━━ 948s 61ms/step - accuracy: 0.4300 - loss: 1.9233


2026/03/15 01:30:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


saved model with name:  lenet_baseline_chunk_4_2013-01


In [6]:
from pathlib import Path

if mlflow_run_id is None:
    raise RuntimeError("MLflow run was not started. Please re-run the training cell.")

run_id_path = Path('/kaggle/working/mlflow_run_id.txt')
run_id_path.write_text(mlflow_run_id)
print(f"Saved MLflow run id {mlflow_run_id} to {run_id_path}")
mlflow_run_id

Saved MLflow run id 8d31d1cdd302476a9444455441b6673a to /kaggle/working/mlflow_run_id.txt


'8d31d1cdd302476a9444455441b6673a'